# Lesson 2: Data Engineering — Homework

В реальному світі документи бувають зламані, з неправильним кодуванням, порожні, з підміненими розширеннями. Ваша задача — навчитись їх обробляти.

**6 завдань.** Кожне має `# TODO` — дописуйте реалізацію.

```bash
pip install -r requirements.txt
```

In [ ]:
from pathlib import Path

print("Файли для роботи:")
for f in sorted(Path("samples/enterprise_challenges").iterdir()):
    size = f.stat().st_size
    print(f"  {f.name:<40} {size:>10,} bytes")

---
## Завдання 1: Визначення кодування файлів

Legacy-системи часто віддають файли в Windows-1251, Latin-1, UTF-8 з BOM — без вказання charset.
Якщо відкрити такий файл з неправильним кодуванням, отримаємо кракозябри (mojibake).

**Файли для роботи:**
- `utf8_with_bom.html` — UTF-8 з BOM маркером (зайві байти `\xef\xbb\xbf` на початку)
- `windows1251_no_charset.html` — Windows-1251 без мета-тегу charset (українські legacy-системи)
- `latin1_mixed.html` — Latin-1 з німецькими/французькими символами

**Що потрібно зробити:**
1. Прочитати файл як bytes
2. Визначити кодування через `charset_normalizer`
3. Декодувати текст правильно
4. Якщо є BOM — прибрати його

In [ ]:
from charset_normalizer import from_bytes


def detect_and_read(file_path: str) -> dict:
    file_path = Path(file_path)
    raw = file_path.read_bytes()

    had_bom = raw.startswith(b"\xef\xbb\xbf")
    if had_bom:
        raw = raw[3:]

    result = from_bytes(raw).best()
    if result is not None:
        encoding = result.encoding
        text = str(result)
    else:
        encoding = "utf-8"
        text = raw.decode("utf-8", errors="replace")

    return {
        "file": file_path.name,
        "encoding": encoding,
        "had_bom": had_bom,
        "text": text,
        "char_count": len(text),
    }

In [ ]:
# Тест
encoding_files = [
    "samples/enterprise_challenges/utf8_with_bom.html",
    "samples/enterprise_challenges/windows1251_no_charset.html",
    "samples/enterprise_challenges/latin1_mixed.html",
]

for f in encoding_files:
    result = detect_and_read(f)
    print(f"{result['file']}:")
    print(f"  Кодування: {result['encoding']}")
    print(f"  BOM: {result['had_bom']}")
    print(f"  Перші 150 символів: {result['text'][:150]}")
    print()

---
## Завдання 2: Визначення реального типу файлу (magic bytes)

Розширення файлу може брехати. Хтось зберіг HTML як `.pdf`, або binary dump перейменував в `.pdf`.
Єдиний надійний спосіб — перевірити **magic bytes** (перші байти файлу, які визначають формат).

**Файли для роботи:**
- `actually_html.pdf` — HTML контент зі розширенням .pdf
- `actually_pdf.html` — PDF контент зі розширенням .html
- `binary_garbage.pdf` — рандомні байти з розширенням .pdf
- `empty_file.pdf` — порожній файл (0 байт)

**Що потрібно зробити:**
1. Визначити тип файлу по розширенню (declared type)
2. Визначити реальний тип через бібліотеку `filetype` (magic bytes)
3. Для HTML файлів (без magic bytes) — перевірити чи контент починається з `<html` або `<!doctype`
4. Повернути чи є mismatch

In [ ]:
import filetype as filetype_lib


def detect_file_type(file_path: str) -> dict:
    file_path = Path(file_path)
    declared_type = file_path.suffix.lstrip(".").lower()

    if file_path.stat().st_size == 0:
        return {
            "file": file_path.name,
            "declared_type": declared_type,
            "detected_type": None,
            "is_mismatch": True,
            "issue": "empty file",
        }

    guess = filetype_lib.guess(str(file_path))
    if guess is not None:
        detected_type = guess.extension
    else:
        head = file_path.read_bytes()[:512].lower()
        if b"<html" in head or b"<!doctype" in head:
            detected_type = "html"
        else:
            detected_type = None

    is_mismatch = detected_type is not None and detected_type != declared_type
    issue = f"declared .{declared_type} but detected {detected_type}" if is_mismatch else None

    return {
        "file": file_path.name,
        "declared_type": declared_type,
        "detected_type": detected_type,
        "is_mismatch": is_mismatch,
        "issue": issue,
    }

In [ ]:
# Тест
test_files = [
    "samples/enterprise_challenges/actually_html.pdf",
    "samples/enterprise_challenges/actually_pdf.html",
    "samples/enterprise_challenges/binary_garbage.pdf",
    "samples/enterprise_challenges/empty_file.pdf",
    "samples/enterprise_challenges/normal_report.xlsx",
]

for f in test_files:
    if Path(f).exists():
        result = detect_file_type(f)
        marker = "MISMATCH" if result["is_mismatch"] else "OK"
        print(f"  [{marker}] {result['file']}: declared=.{result['declared_type']}, real={result['detected_type']}")
        if result["issue"]:
            print(f"          -> {result['issue']}")

---
## Завдання 3: Витягування тексту з брудного HTML

Enterprise HTML — це жах: 50 рівнів вкладеності від Word-експорту, navigation bars, sidebars, scripts, cookie banners. Корисного тексту — 5%.

**Файли для роботи:**
- `malformed_deeply_nested.html` — 50 рівнів `<div>`, незакриті теги, зламані entities
- `boilerplate_heavy.html` — 30 nav items, 20 sidebar widgets, 3 абзаци корисного тексту

**Що потрібно зробити:**
1. Прочитати HTML через BeautifulSoup
2. Видалити `<script>`, `<style>`, `<nav>`, `<header>`, `<footer>` теги
3. Витягти чистий текст
4. Порахувати "корисність" — відсоток тексту від розміру HTML

In [ ]:
from bs4 import BeautifulSoup


def extract_clean_text(file_path: str) -> dict:
    file_path = Path(file_path)
    raw_html = file_path.read_text(errors="replace")

    soup = BeautifulSoup(raw_html, "html.parser")

    for tag_name in ["script", "style", "nav", "header", "footer", "aside"]:
        for tag in soup.find_all(tag_name):
            tag.decompose()

    text = soup.get_text(separator="\n", strip=True)

    raw_size = len(raw_html)
    text_size = len(text)
    useful_ratio = text_size / raw_size if raw_size > 0 else 0

    return {
        "file": file_path.name,
        "raw_size": raw_size,
        "text": text,
        "text_size": text_size,
        "useful_ratio": useful_ratio,
    }

In [ ]:
# Тест
html_files = [
    "samples/enterprise_challenges/malformed_deeply_nested.html",
    "samples/enterprise_challenges/boilerplate_heavy.html",
    "samples/enterprise_challenges/multilingual.html",
]

for f in html_files:
    result = extract_clean_text(f)
    print(f"{result['file']}:")
    print(f"  HTML: {result['raw_size']:,} символів -> Текст: {result['text_size']:,} символів")
    print(f"  Корисність: {result['useful_ratio']:.1%}")
    print(f"  Текст: {result['text'][:200]}...")
    print()

---
## Завдання 4: Обробка зламаних файлів — safe parser

Файли ламаються: обрізані PDF, corrupted DOCX (зламаний ZIP), binary garbage з розширенням .pdf, порожні файли, захищені паролем PDF.

Парсер не повинен падати — він має повертати результат або зрозумілу помилку.

**Файли для роботи:** вся папка `enterprise_challenges/` — там є і робочі, і зламані файли.

**Що потрібно зробити:**
1. Перевірити файл (порожній? правильний тип?)
2. Спробувати спарсити через `unstructured.partition.auto.partition`
3. Якщо помилка — зловити exception і повернути опис проблеми
4. Класифікувати помилку: `empty`, `corrupted`, `type_mismatch`, `parse_error`

In [ ]:
from unstructured.partition.auto import partition


def safe_parse(file_path: str) -> dict:
    file_path = Path(file_path)

    if file_path.stat().st_size == 0:
        return {"file": file_path.name, "status": "error",
                "error_type": "empty", "error_message": "file is empty"}

    file_info = detect_file_type(str(file_path))
    if file_info["is_mismatch"]:
        return {"file": file_path.name, "status": "error",
                "error_type": "type_mismatch",
                "error_message": file_info.get("issue", "type mismatch")}

    try:
        elements = partition(filename=str(file_path))
        text = "\n".join(str(el) for el in elements)
        return {"file": file_path.name, "status": "ok",
                "text": text, "char_count": len(text)}
    except Exception as e:
        return {"file": file_path.name, "status": "error",
                "error_type": "corrupted", "error_message": str(e)}

In [ ]:
# Тест: прогнати safe_parse по всій папці enterprise_challenges
results = []
challenges_dir = Path("samples/enterprise_challenges")

for f in sorted(challenges_dir.iterdir()):
    if f.is_file():
        result = safe_parse(str(f))
        results.append(result)

ok = [r for r in results if r["status"] == "ok"]
errors = [r for r in results if r["status"] == "error"]

print(f"=== Результати: {len(ok)} OK, {len(errors)} ERROR ===\n")

print("OK:")
for r in ok:
    print(f"  {r['file']}: {r['char_count']} символів")

print(f"\nERROR:")
for r in errors:
    print(f"  {r['file']}: [{r['error_type']}] {r['error_message']}")

---
## Завдання 5: Витягування таблиць з PDF

`unstructured` витягує текст з PDF, але **губить структуру таблиць** — рядки і колонки перемішуються в суцільний текст. Для таблиць потрібен спеціальний інструмент — `pdfplumber`.

**Файл для роботи:**
- `financial_report_table.pdf` — фінансовий звіт з двома таблицями (revenue by region, revenue by product)

**Що потрібно зробити:**
1. Витягти таблиці з PDF через `pdfplumber`
2. Перетворити кожну таблицю в список словників (перший рядок — ключі)
3. Порівняти результат з тим що дає `unstructured` — побачити різницю

In [ ]:
import pdfplumber


def extract_tables_from_pdf(file_path: str) -> list[list[dict]]:
    all_tables = []

    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages:
            for raw_table in page.extract_tables():
                headers = raw_table[0]
                table = [dict(zip(headers, row)) for row in raw_table[1:]]
                all_tables.append(table)

    return all_tables

In [ ]:
pdf_path = "samples/enterprise_challenges/financial_report_table.pdf"

# --- pdfplumber: структуровані таблиці ---
tables = extract_tables_from_pdf(pdf_path)
print(f"pdfplumber знайшов {len(tables)} таблиць\n")

for i, table in enumerate(tables):
    print(f"=== Таблиця {i+1}: {len(table)} рядків ===")
    if table:
        print(f"  Колонки: {list(table[0].keys())}")
        for row in table[:3]:
            print(f"  {row}")
        if len(table) > 3:
            print(f"  ... ще {len(table) - 3} рядків")
    print()

# --- unstructured: для порівняння ---
print("=== unstructured (для порівняння) ===")
elements = partition(filename=pdf_path)
text = "\n".join(str(el) for el in elements)
print(text[:500])
print("\n^ Бачите різницю? unstructured втрачає структуру таблиці.")

---
## Завдання 6: Chunking великого документа

Для RAG та embeddings текст треба розбити на чанки. Але з великим файлом (1MB+) є нюанси:
- Як розмір чанка впливає на кількість чанків?
- Що відбувається з overlap?
- Скільки часу займає chunking?

**Файл для роботи:**
- `huge_audit_log.txt` — ~1.5 MB, 5000 записів audit log

**Що потрібно зробити:**
1. Реалізувати chunking через `RecursiveCharacterTextSplitter`
2. Порівняти різні `chunk_size` (256, 512, 1024, 2048) — кількість чанків, час
3. Порівняти різні `chunk_overlap` (0, 50, 200) — як overlap впливає на кількість чанків

In [ ]:
huge_file = Path("samples/enterprise_challenges/huge_audit_log.txt")
text = huge_file.read_text()
size_mb = huge_file.stat().st_size / 1024 / 1024
print(f"Файл: {huge_file.name}")
print(f"Розмір: {size_mb:.2f} MB, {len(text):,} символів")

In [ ]:
import time
from langchain_text_splitters import RecursiveCharacterTextSplitter


def chunk_text(text: str, chunk_size: int = 512, chunk_overlap: int = 50) -> list[str]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap
    )
    return splitter.split_text(text)

In [ ]:
# Експеримент 1: різний chunk_size (overlap=50)
print(f"Текст: {len(text):,} символів\n")
print(f"{'chunk_size':>12} | {'чанків':>8} | {'сер. довж.':>10} | {'час (мс)':>10}")
print("-" * 52)

for size in [256, 512, 1024, 2048]:
    t0 = time.time()
    chunks = chunk_text(text, chunk_size=size, chunk_overlap=50)
    elapsed = (time.time() - t0) * 1000
    avg = sum(len(c) for c in chunks) / len(chunks) if chunks else 0
    print(f"{size:>12} | {len(chunks):>8} | {avg:>10.0f} | {elapsed:>10.1f}")

In [ ]:
# Експеримент 2: різний chunk_overlap (chunk_size=512)
print(f"chunk_size=512, текст: {len(text):,} символів\n")
print(f"{'overlap':>12} | {'чанків':>8} | {'додатково':>12} | {'час (мс)':>10}")
print("-" * 52)

baseline = None
for overlap in [0, 50, 100, 200]:
    t0 = time.time()
    chunks = chunk_text(text, chunk_size=512, chunk_overlap=overlap)
    elapsed = (time.time() - t0) * 1000
    if baseline is None:
        baseline = len(chunks)
    extra = len(chunks) - baseline
    print(f"{overlap:>12} | {len(chunks):>8} | +{extra:>10} | {elapsed:>10.1f}")

print(f"\n(overlap збільшує кількість чанків, бо текст дублюється на стиках)")

In [ ]:
# Подивимось на стик між чанками — як працює overlap
chunks = chunk_text(text, chunk_size=512, chunk_overlap=100)
print(f"Всього чанків: {len(chunks)}\n")

print("=== Чанк 0 (останні 150 символів) ===")
print(f"...{chunks[0][-150:]}")
print(f"\n=== Чанк 1 (перші 150 символів) ===")
print(f"{chunks[1][:150]}...")
print(f"\n^ Бачите overlap? Кінець чанка 0 повторюється на початку чанка 1.")
print("  Це потрібно щоб контекст не губився на стиках.")

---
## Готово!

Ви навчились:
1. **Визначати кодування** — charset_normalizer, BOM, legacy encodings
2. **Перевіряти тип файлу** — magic bytes vs розширення
3. **Чистити HTML** — видаляти шум, витягувати корисний текст
4. **Безпечно парсити** — обробляти corrupted/empty/broken файли без crash
5. **Витягувати таблиці з PDF** — pdfplumber vs unstructured, структуровані дані
6. **Chunking** — як chunk_size і overlap впливають на результат для великих документів